# CNN runs → `runs/` folders (mouse vs Sensorium)

Same visual-only CNN (`PredictorTransfer`, shifter off by default) on either dataset. Every run writes `runs/<timestamp>_<name>/` with `config.json`, `metrics.csv`, `summary.json`, `best_val.pth` and per-neuron arrays. Plot them with `plot_runs.ipynb`.

- **Mouse:** `mouse_time_block_split`: ~12 s blocks go wholly to train or val, with a gap around val blocks.
- **Sensorium:** `SensoriumCachedDataset` (one sample per real 36×64 frame → 60×80) + `sensorium_trial_split`: whole trials go to train or val, over **all** trials (competition tiers ignored).
- Training-size sweeps are specified in **minutes**; subsets take whole blocks / trials, and the val set is the same for every size.
- Small training sets get enough epochs to reach `MIN_STEPS` gradient steps (early stopping still applies), so size isn't confounded with the number of updates.

In [ ]:
import math
import os

import numpy as np
import torch

from mouse_model.cnn_predictor_transfer import (
    PredictorTransfer,
    freeze_encoder_backbone,
    load_encoder_backbone_from_checkpoint,
)
from mouse_model.run_logging import RunLogger, load_runs
from mouse_model.sensorium_cache import SensoriumCachedDataset, build_sensorium_cache
from mouse_model.splits import MOUSE_FRAME_S, mouse_time_block_split, sensorium_trial_split
from mouse_model.transfer_cnn_training import train_cnn_transfer

SENSORIUM_ROOT = os.path.expanduser("~/Downloads/dynamic29515-10-12-Video-9b4f6a1a067fe51e15306b9628efea20")
MOUSE_FILE_ID = "070921_J553RT"

SEED = 0
BATCH_SIZE = 256
LEARNING_RATE = 1e-4
MIN_EPOCHS = 100
MIN_STEPS = 3000
MAX_EPOCHS = 1000
PATIENCE = 5
VAL_FRAC = 0.3
MOUSE_BLOCK_LEN = 250  # samples (12 s at 48 ms)
SENSORIUM_RESPONSE_LAG = 1  # frames (33 ms each); mouse uses predict_offset=1 (48 ms)
USE_SHIFTER = False
NUM_WORKERS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SENSORIUM_CACHE = build_sensorium_cache(SENSORIUM_ROOT)  # one-time, ~2 min per session
sensorium_ds = SensoriumCachedDataset(SENSORIUM_CACHE, response_lag=SENSORIUM_RESPONSE_LAG)
print("Sensorium frames:", len(sensorium_ds), f"({len(sensorium_ds) / sensorium_ds.fps / 60:.0f} min)")

In [ ]:
def minutes_to_samples(dataset, minutes):
    if minutes is None:
        return None
    per_min = 60.0 / MOUSE_FRAME_S if dataset == "mouse" else 60.0 * sensorium_ds.fps
    return int(round(minutes * per_min))


def load_data(dataset, max_train_samples, seed):
    if dataset == "mouse":
        tr, va, info = mouse_time_block_split(
            MOUSE_FILE_ID, block_len=MOUSE_BLOCK_LEN, val_frac=VAL_FRAC, seed=seed,
            max_train_samples=max_train_samples,
        )
        return tr, va, info, info["num_neurons"]
    tr, va, info = sensorium_trial_split(
        sensorium_ds, val_frac=VAL_FRAC, seed=seed, max_train_samples=max_train_samples,
    )
    info["response_lag"] = SENSORIUM_RESPONSE_LAG
    info["cache_dir"] = SENSORIUM_CACHE
    return tr, va, info, sensorium_ds.num_neurons


def epochs_for(n_train):
    steps_per_epoch = max(1, math.ceil(n_train / BATCH_SIZE))
    return int(min(MAX_EPOCHS, max(MIN_EPOCHS, math.ceil(MIN_STEPS / steps_per_epoch))))


def already_done(**cfg):
    return any(r["summary"] for r in load_runs(**cfg))


def run_one(dataset, train_minutes=None, seed=SEED, training_mode="from_scratch",
            pretrained_run_dir=None, skip_existing=True):
    max_train = minutes_to_samples(dataset, train_minutes)
    key = dict(dataset=dataset, train_minutes_requested=train_minutes, seed=seed,
               training_mode=training_mode, shifter=USE_SHIFTER,
               pretrained_run=None if pretrained_run_dir is None else os.path.basename(pretrained_run_dir))
    if skip_existing and already_done(**key):
        print("skip (done):", key)
        return None

    tr, va, info, n_neurons = load_data(dataset, max_train, seed)
    model = PredictorTransfer(num_neurons=n_neurons, use_shifter=USE_SHIFTER).to(device)
    frozen = training_mode == "transfer_frozen_backbone"
    if pretrained_run_dir is not None:
        sd = torch.load(os.path.join(pretrained_run_dir, "best_val.pth"), map_location=device)
        load_encoder_backbone_from_checkpoint(model, sd)
        if frozen:
            freeze_encoder_backbone(model)

    tag = "full" if train_minutes is None else f"{train_minutes}min"
    run = RunLogger(
        f"{dataset}_{training_mode}_{tag}_s{seed}",
        {**key, "model": "cnn", "max_train_samples": max_train, "split_info": info},
    )
    print(run.dir, f"| train {info['n_train']} ({info['train_minutes']:.1f} min) val {info['n_val']}")
    train_cnn_transfer(
        model, device, tr, va,
        epochs=epochs_for(info["n_train"]), batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
        seed=seed, num_workers=NUM_WORKERS, freeze_encoder_backbone=frozen, patience=PATIENCE, run=run,
    )
    return run.dir

## Learning-curve sweep (scratch)

Mouse `070921_J553RT` has ~20 min of training data under this split; one Sensorium session has ~80 min. `None` = all training data. Re-running the cell skips finished runs.

In [ ]:
TRAIN_MINUTES = [1, 2, 5, 10, 20, None]
SEEDS = [0]

for seed in SEEDS:
    for dataset in ["mouse", "sensorium"]:
        for minutes in TRAIN_MINUTES:
            run_one(dataset, minutes, seed=seed)

## Transfer (backbone from a finished full-data scratch run)

In [ ]:
def full_scratch_run(dataset, seed=SEED):
    runs = [r for r in load_runs(dataset=dataset, training_mode="from_scratch", seed=seed,
                                 train_minutes_requested=None) if r["summary"]]
    return runs[-1]["dir"] if runs else None


for target, source in [("sensorium", "mouse"), ("mouse", "sensorium")]:
    src = full_scratch_run(source)
    if src is None:
        print(f"no finished full {source} scratch run yet")
        continue
    for mode in ["transfer_frozen_backbone", "transfer_finetune"]:
        run_one(target, None, training_mode=mode, pretrained_run_dir=src)